# Creating and Valuing a CDS Contract

Creates a CDS contract and does a valuation and calculates risk measures

In [1]:
import numpy as np
import matplotlib.pyplot as plt

In [2]:
from financepy.utils.date import Date
from financepy.utils.global_vars import ONE_MILLION
from financepy.utils.day_count import DayCountTypes
from financepy.utils.frequency import FrequencyTypes

from financepy.products.rates.ibor_deposit import IborDeposit
from financepy.products.rates.ibor_swap import IborSwap, SwapTypes
from financepy.products.credit.cds import CDS

from financepy.market.curves.flat_discount_curve import FlatDiscountCurve
from financepy.market.curves.cds_curve import CDSCurve
from financepy.market.curves.ibor_single_curve import IborSingleCurve

#############################################################
#  FINANCEPY Version 1.1.2 - Built on 25 Sep 2026 at 14:07  #
#  This software is distributed FREE AND WITHOUT WARRANTY   #
#  Report issues at https://github.com/domokane/FinancePy   #
#############################################################



## Creating a CDS Contract

In [3]:
value_dt = Date(10, 1, 2026)
effective_dt = Date(1,12,2024)
maturity_dt = Date(20, 3, 2028)
cds_coupon = 0.010
notional = ONE_MILLION
long_protection = True
trade_dt = Date(9, 8, 2019)

In [4]:
interest_rate = 0.05
discount_curve = FlatDiscountCurve(value_dt, interest_rate)

In [5]:
cds_contract = CDS(effective_dt, maturity_dt, cds_coupon, notional, long_protection)

In [6]:
print(cds_contract)

OBJECT_TYPE: CDS
STEP_IN_DATE: 01-DEC-2024
MATURITY: 20-MAR-2028
NOTIONAL: 1000000
LONG_PROT: True
COUPON: 100.0bp
DAY_COUNT: DayCountTypes.ACT_360
FREQUENCY: FrequencyTypes.QUARTERLY
CALENDAR: CalendarTypes.WEEKEND
BUS_DAY_ADJUST: BusDayAdjustTypes.FOLLOWING
DATE_GEN_RULE: DateGenRuleTypes.BACKWARD
PAYMENT_DT, YEAR_FRAC, ACCRUAL_START, ACCRUAL_END, PAYMENT
20-DEC-2024,     0.252778, 20-SEP-2024, 19-DEC-2024,  2527.777778
20-MAR-2025,     0.250000, 20-DEC-2024, 19-MAR-2025,  2500.000000
20-JUN-2025,     0.255556, 20-MAR-2025, 19-JUN-2025,  2555.555556
22-SEP-2025,     0.261111, 20-JUN-2025, 21-SEP-2025,  2611.111111
22-DEC-2025,     0.252778, 22-SEP-2025, 21-DEC-2025,  2527.777778
20-MAR-2026,     0.244444, 22-DEC-2025, 19-MAR-2026,  2444.444444
22-JUN-2026,     0.261111, 20-MAR-2026, 21-JUN-2026,  2611.111111
21-SEP-2026,     0.252778, 22-JUN-2026, 20-SEP-2026,  2527.777778
21-DEC-2026,     0.252778, 21-SEP-2026, 20-DEC-2026,  2527.777778
22-MAR-2027,     0.252778, 21-DEC-2026, 21-MAR

# Build a CDS Curve

In [7]:
settle_dt = value_dt.add_days(1)

In [8]:
cds1 = CDS(settle_dt, "1Y", 0.0065)
cds2 = CDS(settle_dt, "2Y", 0.0070)
cds3 = CDS(settle_dt, "3Y", 0.0075)
cds5 = CDS(settle_dt, "5Y", 0.0080)

In [9]:
print(cds2)

OBJECT_TYPE: CDS
STEP_IN_DATE: 11-JAN-2026
MATURITY: 20-MAR-2028
NOTIONAL: 1000000
LONG_PROT: True
COUPON: 70.0bp
DAY_COUNT: DayCountTypes.ACT_360
FREQUENCY: FrequencyTypes.QUARTERLY
CALENDAR: CalendarTypes.WEEKEND
BUS_DAY_ADJUST: BusDayAdjustTypes.FOLLOWING
DATE_GEN_RULE: DateGenRuleTypes.BACKWARD
PAYMENT_DT, YEAR_FRAC, ACCRUAL_START, ACCRUAL_END, PAYMENT
20-MAR-2026,     0.244444, 22-DEC-2025, 19-MAR-2026,  1711.111111
22-JUN-2026,     0.261111, 20-MAR-2026, 21-JUN-2026,  1827.777778
21-SEP-2026,     0.252778, 22-JUN-2026, 20-SEP-2026,  1769.444444
21-DEC-2026,     0.252778, 21-SEP-2026, 20-DEC-2026,  1769.444444
22-MAR-2027,     0.252778, 21-DEC-2026, 21-MAR-2027,  1769.444444
21-JUN-2027,     0.252778, 22-MAR-2027, 20-JUN-2027,  1769.444444
20-SEP-2027,     0.252778, 21-JUN-2027, 19-SEP-2027,  1769.444444
20-DEC-2027,     0.252778, 20-SEP-2027, 19-DEC-2027,  1769.444444
20-MAR-2028,     0.255556, 20-DEC-2027, 20-MAR-2028,  1788.888889


In [10]:
cds_list = [cds1, cds2, cds3, cds5]

In [11]:
recovery_rate = 0.40

In [12]:
cds_curve = CDSCurve(value_dt, cds_list, discount_curve, recovery_rate)

In [13]:
print(cds_curve)

OBJECT_TYPE: CDSCurve
TIME,SURVIVAL_PROBABILITY
 0.0000000,  1.0000000
 1.1890411,  0.9868964
 2.1917808,  0.9742773
 3.1917808,  0.9601136
 5.1917808,  0.9316923


# Valuation

In [14]:
cds_contract.print_payments(value_dt, cds_curve)

PAYMENT_DT      YEAR_FRAC      PAYMENT           DF       SURV_PROB      NPV
    20-MAR-2026   0.244444      2444.44     0.990592     0.997905      2416.38
    22-JUN-2026   0.261111      2611.11     0.977919     0.995058      2540.84
    21-SEP-2026   0.252778      2527.78     0.965804     0.992310      2422.56
    21-DEC-2026   0.252778      2527.78     0.953839     0.989570      2385.94
    22-MAR-2027   0.252778      2527.78     0.942023     0.986827      2349.86
    21-JUN-2027   0.252778      2527.78     0.930352     0.983675      2313.33
    20-SEP-2027   0.252778      2527.78     0.918827     0.980532      2277.37
    20-DEC-2027   0.252778      2527.78     0.907444     0.977400      2241.98
    20-MAR-2028   0.255556      2555.56     0.896202     0.974277      2231.38


In [15]:
cds_contract.rpv01(value_dt, cds_curve)


(np.float64(2.1375836153526078), np.float64(2.08480583757483))

In [16]:
spd = cds_contract.par_spread(value_dt, cds_curve, recovery_rate) * 10000.0
print("FAIR CDS SPREAD %10.5f bp"% spd)

FAIR CDS SPREAD   69.99413 bp


In [17]:
v = cds_contract.value(value_dt, cds_curve, recovery_rate)

In [18]:
dirty_pv = v[0]
clean_pv = v[1]

In [19]:
print("DIRTY VALUE %12.2f"% dirty_pv)
print("CLEAN VALUE %12.2f"% clean_pv)

DIRTY VALUE     -6783.42
CLEAN VALUE     -6255.64


In [20]:
cleanp = cds_contract.clean_price(settle_dt, cds_curve, recovery_rate)
print("CLEAN PRICE %12.6f"% cleanp)

CLEAN PRICE   100.624990


In [21]:
accrued_days = cds_contract.accrued_days(value_dt)
print("ACCRUED_DAYS", accrued_days)

ACCRUED_DAYS 19.0


In [22]:
accrued_interest = cds_contract.accrued_interest(value_dt)
print("ACCRUED_COUPON", accrued_interest)

ACCRUED_COUPON 527.7777777777778


In [23]:
prot_pv = cds_contract.prot_leg_pv(settle_dt, cds_curve, recovery_rate)
print("prot_PV", prot_pv)

prot_PV 14573.992799996993


In [24]:
premPV = cds_contract.premium_leg_pv(settle_dt, cds_curve, recovery_rate)
print("PREMIUM_PV", premPV)

PREMIUM_PV 21379.444350940284


In [25]:
rpv01 = cds_contract.rpv01(settle_dt, cds_curve)
print("DIRTY_RPV01", rpv01[0])
print("CLEAN_RPV01", rpv01[1])

DIRTY_RPV01 2.1379444350940284
CLEAN_RPV01 2.082388879538473


In [26]:
cds_contract.rpv01(settle_dt, cds_curve)

(np.float64(2.1379444350940284), np.float64(2.082388879538473))

In [27]:
cds_contract.value_fast_approx(value_dt, interest_rate, 0.070, recovery_rate)

(110979.56089677893,
 111507.33867455671,
 1.9112334223537226,
 1.8584556445759448,
 166.43262718016922,
 -11.469648639540537,
 -229.64135526577593)

## Risk Measures

In [28]:
spread_dv01 = cds_contract.spread_dv01(settle_dt, cds_curve, recovery_rate)

In [29]:
spread_dv01

np.float64(209.33448763444358)

In [30]:
2.25*0.003*1000000

6750.0

Copyright (c) 2020 Dominic O'Kane